In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

In [2]:
bronze_path = Path.cwd()/"data"/"bronze"
    
#file_name = "tfl-bikes-2026-07-24-H12-M45.json"


dataframes = []

for file in list(bronze_path.iterdir()):
    
    # if len(dataframes)>5:
    #     print("BREAK")
    #     break
        
    if file.stat().st_size == 0:
        print(f"Skipping empty file: {file.name}")
        continue
        
    if ((file.name[:10]+file.name[-5:])!="tfl-bikes-.json"):
        print(f"Formatting error using {file.name}")
        raise ValueError
        
    else:
        time = (file.name[10:-5])
        date_time = datetime.strptime(
            time,
            "%Y-%m-%d-H%H-M%M"
        )
        
    with open(file) as f:
        json_input = json.load(f)
    
    df = pd.json_normalize(json_input)
    df["date_time"] = date_time

    dataframes.append(df)

print(len(dataframes))

Skipping empty file: tfl-bikes-2026-07-24-H13-M45.json
Skipping empty file: tfl-bikes-2026-07-24-H13-M50.json
Skipping empty file: tfl-bikes-2026-07-24-H13-M55.json
Skipping empty file: tfl-bikes-2026-07-24-H14-M15.json
1713


In [3]:
df = pd.concat(dataframes, ignore_index=True)

In [4]:
df_long = df.explode("additionalProperties",True)

In [5]:
properties = pd.json_normalize(
    df_long["additionalProperties"]
)

In [6]:
properties

,$type,category,key,sourceSystemKey,value,modified
0,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,TerminalName,BikePoints,001023,2026-07-24T12:43:59.477Z
1,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,Installed,BikePoints,true,2026-07-24T12:43:59.477Z
2,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,Locked,BikePoints,false,2026-07-24T12:43:59.477Z
3,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,InstallDate,BikePoints,1278947280000,2026-07-24T12:43:59.477Z
4,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,RemovalDate,BikePoints,,2026-07-24T12:43:59.477Z
...,...,...,...,...,...,...
15033590,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,NbBikes,BikePoints,1,2026-07-30T10:30:52.107Z
15033591,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,NbEmptyDocks,BikePoints,15,2026-07-30T10:30:52.107Z
15033592,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,NbDocks,BikePoints,16,2026-07-30T10:30:52.107Z
15033593,Tfl.Api.Presentation.Entities.AdditionalProper...,Description,NbStandardBikes,BikePoints,1,2026-07-30T10:30:52.107Z


In [7]:
df_long

,$type,id,url,commonName,placeType,additionalProperties,children,childrenUrls,lat,lon,date_time
0,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_1,/Place/BikePoints_1,"River Street , Clerkenwell",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.529163,-0.109970,2026-07-24 12:45:00
1,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_1,/Place/BikePoints_1,"River Street , Clerkenwell",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.529163,-0.109970,2026-07-24 12:45:00
2,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_1,/Place/BikePoints_1,"River Street , Clerkenwell",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.529163,-0.109970,2026-07-24 12:45:00
3,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_1,/Place/BikePoints_1,"River Street , Clerkenwell",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.529163,-0.109970,2026-07-24 12:45:00
4,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_1,/Place/BikePoints_1,"River Street , Clerkenwell",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.529163,-0.109970,2026-07-24 12:45:00
...,...,...,...,...,...,...,...,...,...,...,...
15033590,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_403,/Place/BikePoints_403,"George Place Mews, Marylebone",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.516892,-0.158249,2026-07-30 11:45:00
15033591,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_403,/Place/BikePoints_403,"George Place Mews, Marylebone",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.516892,-0.158249,2026-07-30 11:45:00
15033592,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_403,/Place/BikePoints_403,"George Place Mews, Marylebone",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.516892,-0.158249,2026-07-30 11:45:00
15033593,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_403,/Place/BikePoints_403,"George Place Mews, Marylebone",BikePoint,{'$type': 'Tfl.Api.Presentation.Entities.Addit...,[],[],51.516892,-0.158249,2026-07-30 11:45:00


In [8]:
dfx = pd.concat(
    [
        df_long[["id", "date_time"]],
        properties
    ],
    axis=1
)

In [9]:
dfx_filtered = dfx[["id", "date_time", "key", "value"]]
dfx_filtered

,id,date_time,key,value
0,BikePoints_1,2026-07-24 12:45:00,TerminalName,001023
1,BikePoints_1,2026-07-24 12:45:00,Installed,true
2,BikePoints_1,2026-07-24 12:45:00,Locked,false
3,BikePoints_1,2026-07-24 12:45:00,InstallDate,1278947280000
4,BikePoints_1,2026-07-24 12:45:00,RemovalDate,
...,...,...,...,...
15033590,BikePoints_403,2026-07-30 11:45:00,NbBikes,1
15033591,BikePoints_403,2026-07-30 11:45:00,NbEmptyDocks,15
15033592,BikePoints_403,2026-07-30 11:45:00,NbDocks,16
15033593,BikePoints_403,2026-07-30 11:45:00,NbStandardBikes,1


In [10]:
dfx_filtered["key"].unique()

array(['TerminalName', 'Installed', 'Locked', 'InstallDate',
       'RemovalDate', 'Temporary', 'NbBikes', 'NbEmptyDocks', 'NbDocks',
       'NbStandardBikes', 'NbEBikes', 'LastUpdated', nan], dtype=object)

In [11]:
wanted = dfx_filtered["key"].unique()[6:11]

In [12]:
dfx_filtered

,id,date_time,key,value
0,BikePoints_1,2026-07-24 12:45:00,TerminalName,001023
1,BikePoints_1,2026-07-24 12:45:00,Installed,true
2,BikePoints_1,2026-07-24 12:45:00,Locked,false
3,BikePoints_1,2026-07-24 12:45:00,InstallDate,1278947280000
4,BikePoints_1,2026-07-24 12:45:00,RemovalDate,
...,...,...,...,...
15033590,BikePoints_403,2026-07-30 11:45:00,NbBikes,1
15033591,BikePoints_403,2026-07-30 11:45:00,NbEmptyDocks,15
15033592,BikePoints_403,2026-07-30 11:45:00,NbDocks,16
15033593,BikePoints_403,2026-07-30 11:45:00,NbStandardBikes,1


In [13]:
mask = dfx_filtered["key"].isin(wanted)

wanted_rows = dfx_filtered[mask]

wanted_rows#[["id", "key", "value"]]

,id,date_time,key,value
6,BikePoints_1,2026-07-24 12:45:00,NbBikes,2
7,BikePoints_1,2026-07-24 12:45:00,NbEmptyDocks,16
8,BikePoints_1,2026-07-24 12:45:00,NbDocks,19
9,BikePoints_1,2026-07-24 12:45:00,NbStandardBikes,1
10,BikePoints_1,2026-07-24 12:45:00,NbEBikes,1
...,...,...,...,...
15033590,BikePoints_403,2026-07-30 11:45:00,NbBikes,1
15033591,BikePoints_403,2026-07-30 11:45:00,NbEmptyDocks,15
15033592,BikePoints_403,2026-07-30 11:45:00,NbDocks,16
15033593,BikePoints_403,2026-07-30 11:45:00,NbStandardBikes,1


In [14]:
wanted_rows.columns

Index(['id', 'date_time', 'key', 'value'], dtype='object')

In [37]:
df_pivot = dfx_filtered.pivot(index=["id", "date_time"], columns="key", values="value")
wanted_columns = ['NbBikes', 'NbEBikes', 'NbStandardBikes', 'NbDocks', 'NbEmptyDocks']

new_df = df_pivot.loc[:, wanted_columns].copy()

new_df.dtypes

new_df = new_df.astype("Int64")

new_df.head(10)

key                               NbBikes  NbEBikes  NbStandardBikes  NbDocks  \
id           date_time                                                          
BikePoints_1 2026-07-24 12:45:00        2         1                1       19   
             2026-07-24 12:50:00        3         2                1       19   
             2026-07-24 12:55:00        3         2                1       19   
             2026-07-24 13:00:00        3         2                1       19   
             2026-07-24 13:05:00        3         2                1       19   
             2026-07-24 13:10:00        3         2                1       19   
             2026-07-24 13:15:00        3         2                1       19   
             2026-07-24 13:20:00        3         2                1       19   
             2026-07-24 13:25:00        3         2                1       19   
             2026-07-24 13:30:00        3         2                1       19   

key                               NbEmptyDocks  
id           date_time                          
BikePoints_1 2026-07-24 12:45:00            16  
             2026-07-24 12:50:00            16  
             2026-07-24 12:55:00            16  
             2026-07-24 13:00:00            16  
             2026-07-24 13:05:00            16  
             2026-07-24 13:10:00            16  
             2026-07-24 13:15:00            16  
             2026-07-24 13:20:00            16  
             2026-07-24 13:25:00            16  
             2026-07-24 13:30:00            16

In [19]:
# OUTPUT
# 

output_path = Path.cwd() / "data" / "silver" / "tfl-bike-data.xlsx"

output_path.parent.mkdir(parents=True, exist_ok=True)



sample = new_df.head(500000)

sample.reset_index().to_excel(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: C:\Users\jaack\Desktop\cycle-availability-predictor\data\silver\tfl-bike-data.xlsx
